In [1]:
import hail as hl
hl.init(
    spark_conf={
        'spark.hadoop.fs.gs.requester.pays.mode': 'CUSTOM',
        'spark.hadoop.fs.gs.requester.pays.buckets': 'regional_missense_constraint,gnomad-public-requester-pays,gnomad',
        'spark.hadoop.fs.gs.requester.pays.project.id': 'lily-sandbox-a29d'
    }, default_reference='GRCh38'
)

/opt/conda/miniconda3/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning: Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.
  warnings.warn(
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SPARKMONITOR_LISTENER: Started SparkListener for Jupyter Notebook
SPARKMONITOR_LISTENER: Port obtained from environment: 41515
SPARKMONITOR_LISTENER: Application Started: application_1768867195754_0007 ...Start Time: 1768877097891


Running on Apache Spark version 3.3.0
SparkUI available at http://lw-m.us-central1-b.c.lily-sandbox-a29d.internal:44353
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.120-f00f916faf78
LOGGING: writing to /home/hail/hail-20260120-0244-0.2.120-f00f916faf78.log


In [2]:
from rmc.utils.missense_badness import prepare_amino_acid_ht

In [3]:
from gnomad.resources.resource_utils import DataException
from gnomad.utils.file_utils import file_exists

from rmc.resources.basics import MPC_PREFIX, TEMP_PATH_WITH_FAST_DEL
from rmc.resources.reference_data import FOLD_K, train_val_test_transcripts_path
from rmc.resources.resource_utils import CURRENT_GNOMAD_VERSION, KEEP_CODING_CSQ
from rmc.resources.rmc import (
    CURRENT_FREEZE,
    
    amino_acids_oe_path,
    filtered_context,
    misbad_path,
)
from rmc.utils.constraint import (
    add_obs_annotation,
    explode_intervals_to_loci,
    get_constraint_transcripts,
    get_oe_annotation,
)
from rmc.utils.generic import (
    annotate_and_filter_codons,
    filter_context_using_gnomad,
    process_context_ht,
)

In [4]:
from gnomad_constraint.resources.resource_utils import get_preprocessed_ht

In [5]:
from gnomad.utils.constraint import annotate_exploded_vep_for_constraint_groupings
from gnomad.utils.vep import (
    CSQ_NON_CODING,
    explode_by_vep_annotation,
    filter_vep_transcript_csqs,
    process_consequences,
)

# Create variant context table annotated with amino acids, codons, O/E

In [6]:
def process_context_ht_for_misbad(
    filter_to_canonical: bool = False,
    filter_csq = None,
) -> hl.Table:
    """
    Get context HT for SNPs annotated with VEP in canonical protein-coding transcripts
    for missense badness.

    This function offers options to filter to specific variant consequences and add annotations
    to prepare for regional missense constraint calculations.

    :param filter_to_canonical: Whether to filter to canonical transcripts only. Default is False.
    :param filter_csq: Specific consequences to keep. Default is None.
    :return: VEP context HT filtered to canonical transcripts and optionally filtered to variants
        in non-outlier transcripts with specific consequences and annotated with mutation rate etc.
    :rtype: hl.Table
    """
    print("Reading in gene constraint preprocessed context ht...")
    
    ht = hl.read_table(
        "gs://gnomad/v4.1/constraint_coverage_corrected/preprocessed_data/gnomad.v4.1.context.preprocessed.ht"
    ).select_globals()
    ht = ht.annotate(
        exomes_AN_percent=ht.exomes_coverage,
    )
    
    if filter_to_canonical:
        print("Filtering to canonical transcripts only...")

    # Filter to protein-coding ENST transcripts
    # Also optionally filter to canonical transcripts and specific consequences
    ht = filter_vep_transcript_csqs(
        t=ht,
        vep_root="vep",
        synonymous=False,
        canonical=filter_to_canonical,
        protein_coding=True,
        ensembl_only=True,
        filter_empty_csq=True,
        csqs=filter_csq,
    )
    
    print("Exploding VEP annotations...")
    ht = explode_by_vep_annotation(ht, "transcript_consequences")

    # Drop other unnecessary annotations
    return ht.select(
        "context",
        "ref",
        "alt",
        "transcript_consequences",
        "exomes_AN_percent",
    )

In [7]:
context_ht = process_context_ht_for_misbad(filter_csq=KEEP_CODING_CSQ)

Reading in gene constraint preprocessed context ht...


INFO (gnomad.utils.vep 959): Filtering to Ensembl transcripts...
INFO (gnomad.utils.vep 962): Filtering to protein coding transcripts...


Exploding VEP annotations...


In [8]:
context_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'context': str 
    'ref': str 
    'alt': str 
    'transcript_consequences': struct {
        transcript_id: str, 
        gene_id: str, 
        gene_symbol: str, 
        biotype: str, 
        most_severe_consequence: str, 
        mane_select: str, 
        canonical: int32, 
        lof: str, 
        lof_flags: str, 
        sift_score: float64, 
        polyphen_score: float64, 
        domains: array<struct {
            db: str, 
            name: str
        }>, 
        uniprot_isoform: array<str>, 
        amino_acids: str, 
        codons: str
    } 
    'exomes_AN_percent': int32 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [9]:
context_ht.key

<StructExpression of type struct{locus: locus<GRCh38>, alleles: array<str>}>

In [10]:
context_ht.show()

+---------------+------------+---------+-----+-----+
| locus         | alleles    | context | ref | alt |
+---------------+------------+---------+-----+-----+
| locus<GRCh38> | array<str> | str     | str | str |
+---------------+------------+---------+-----+-----+
| chr1:65434    | ["G","A"]  | "ACC"   | "C" | "T" |
| chr1:65434    | ["G","C"]  | "ACC"   | "C" | "G" |
| chr1:65434    | ["G","T"]  | "ACC"   | "C" | "A" |
| chr1:65435    | ["T","A"]  | "TAC"   | "A" | "T" |
| chr1:65435    | ["T","C"]  | "TAC"   | "A" | "G" |
| chr1:65435    | ["T","G"]  | "TAC"   | "A" | "C" |
| chr1:65518    | ["A","C"]  | "CAG"   | "A" | "C" |
| chr1:65518    | ["A","G"]  | "CAG"   | "A" | "G" |
| chr1:65518    | ["A","T"]  | "CAG"   | "A" | "T" |
| chr1:65519    | ["G","A"]  | "ACT"   | "C" | "T" |
+---------------+------------+---------+-----+-----+

+---------------------------------------+---------------------------------+
| transcript_consequences.transcript_id | transcript_consequences.gene_id |
+---------------------------------------+---------------------------------+
| str                                   | str                             |
+---------------------------------------+---------------------------------+
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
| "ENST00000641515"                     | "ENSG00000186092"               |
+---------------------------------------+---------------------------------+

+-------------------------------------+---------------------------------+
| transcript_consequences.gene_symbol | transcript_consequences.biotype |
+-------------------------------------+---------------------------------+
| str                                 | str                             |
+-------------------------------------+---------------------------------+
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
| "OR4F5"                             | "protein_coding"                |
+-------------------------------------+---------------------------------+

+-------------------------------------------------+
| transcript_consequences.most_severe_consequence |
+-------------------------------------------------+
| str                                             |
+-------------------------------------------------+
| "splice_donor_variant"                          |
| "splice_donor_variant"                          |
| "splice_donor_variant"                          |
| "splice_donor_variant"                          |
| "splice_donor_variant"                          |
| "splice_donor_variant"                          |
| "splice_acceptor_variant"                       |
| "splice_acceptor_variant"                       |
| "splice_acceptor_variant"                       |
| "splice_acceptor_va

In [11]:
context_ht = context_ht.select(
    transcript=context_ht.transcript_consequences.transcript_id,
    most_severe_consequence=context_ht.transcript_consequences.most_severe_consequence,
    amino_acids=context_ht.transcript_consequences.amino_acids,
    codons=context_ht.transcript_consequences.codons,
    lof=context_ht.transcript_consequences.lof,
    lof_flags=context_ht.transcript_consequences.lof_flags,
    exomes_AN_percent=context_ht.exomes_AN_percent,
)

In [12]:
pass_transcripts = hl.eval(get_constraint_transcripts(filter_to_canonical=True, outlier=False))

WARNING (regional_missense_constraint_generic 630): Assumes LoF constraint has been separately calculated and that constraint HT exists...


In [13]:
len(pass_transcripts)

17841

In [14]:
# Filter to QC pass transcripts
context_ht = context_ht.filter(
    hl.literal(pass_transcripts).contains(context_ht.transcript)
)

In [15]:
# Filter to high coverage regions
def filt_ht_by_mc_region_median_an(ht, an_threshold = 90):
    median_an_all_mc_intervals_ht = hl.read_table(
        "gs://regional_missense_constraint/temp/all_rmc_intervals_median_AN.ht"
    )
    high_cov_median_an_all_mc_intervals_ht = median_an_all_mc_intervals_ht.filter(
        median_an_all_mc_intervals_ht.median_exomes_AN_percent >= an_threshold
    )
    # Explode to match on both locus and transcript
    expl_high_cov_median_an_all_mc_intervals_ht = explode_intervals_to_loci(
        high_cov_median_an_all_mc_intervals_ht,
        interval_field="interval", keep_intervals=True,
    ).key_by("locus", "transcript")
    # Filter to high coverage regions with median AN greater than threshold
    # over loci used in RMC calculation
    ht = ht.filter(
        hl.is_defined(
            expl_high_cov_median_an_all_mc_intervals_ht[ht.locus, ht.transcript]
        )
    )
    return ht

In [17]:
context_ht = filt_ht_by_mc_region_median_an(context_ht)

In [18]:
context_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'most_severe_consequence': str 
    'amino_acids': str 
    'codons': str 
    'lof': str 
    'lof_flags': str 
    'exomes_AN_percent': int32 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [19]:
context_ht = context_ht.checkpoint(
    f"{TEMP_PATH_WITH_FAST_DEL}/exploded_context_in_hq_transcripts_regions.ht",
    _read_if_exists=False,
    overwrite=True,
)

2026-01-20 03:01:40.564 Hail: INFO: Ordering unsorted dataset with network shuffle
Exception in thread "Thread-38" java.lang.NullPointerException2000 + 76) / 5233]
	at sparkmonitor.listener.JupyterSparkMonitorListener$TaskUpdaterThread.$anonfun$run$1(CustomListener.scala:116)
	at scala.collection.TraversableLike$grouper$1$.apply(TraversableLike.scala:465)
	at scala.collection.TraversableLike$grouper$1$.apply(TraversableLike.scala:455)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at scala.collection.TraversableLike.groupBy(TraversableLike.scala:524)
	at scala.collection.TraversableLike.groupBy$(TraversableLike.scala:454)
	at scala.collection.AbstractTraversable.groupBy(Traversable.scala:108)
	at sparkmonitor.listener.JupyterSparkMonitorListener$TaskUpdaterThread.run(CustomListener.scala:116)
	at java.base/ja

In [20]:
context_ht = annotate_and_filter_codons(context_ht)

INFO (regional_missense_constraint_generic 87): Removing non-coding loci from HT...
INFO (regional_missense_constraint_generic 91): Filtering to lines with expected codon annotations...


In [21]:
loftee_hc_str = "HC"

In [22]:
context_ht.aggregate(hl.agg.counter(context_ht.lof))

{'HC': 3515119, 'LC': 303692, None: 87585322}

In [23]:
context_ht = context_ht.filter(
    (hl.is_missing(context_ht.lof) | (context_ht.lof == loftee_hc_str))    
)

In [24]:
from rmc.utils.generic import get_gnomad_public_release, keep_criteria

In [25]:
def filter_context_using_gnomad_updated(
    context_ht,
    gnomad_data_type: str = "exomes",
    adj_freq_index: int = 0,
    an_pct_threshold: int = 0,
):
    """
    Filter VEP context Table to sites that aren't seen in gnomAD or are rare in gnomAD.

    Also filter sites with zero coverage in gnomAD.

    :param context_ht: VEP context Table.
    :param gnomad_data_type: gnomAD data type. Used to retrieve public release resource.
        Must be one of "exomes" or "genomes" (check is done within `public_release`).
        Default is "exomes".
    :param adj_freq_index: Index of array that contains allele frequency information calculated on
        high quality (adj) genotypes across genetic ancestry groups. Default is 0.
    :param an_threshold: Remove variants at or below this AN threshold (in gnomAD exomes). Default is 0.
    :return: Filtered VEP context Table.
    """
    gnomad = get_gnomad_public_release(gnomad_data_type, adj_freq_index)

    # Filter to sites not seen in gnomAD or to rare sites in gnomAD
    gnomad_join = gnomad[context_ht.key]
    context_ht = context_ht.filter(
        hl.is_missing(gnomad_join)
        | keep_criteria(
            gnomad_join.ac,
            gnomad_join.af,
            gnomad_join.filters,
        )
    )
    return context_ht.filter(
        hl.is_defined(context_ht.exomes_AN_percent) & (context_ht.exomes_AN_percent > an_pct_threshold)
    )

In [26]:
context_ht = filter_context_using_gnomad_updated(context_ht, "exomes")

In [27]:
context_ht = context_ht.checkpoint(
    f"{TEMP_PATH_WITH_FAST_DEL}/rare_coding_exploded_context_in_hq_transcripts_regions.ht",
    _read_if_exists=False,
    overwrite=True,
)

2026-01-20 03:29:10.882 Hail: INFO: wrote table with 86414950 rows in 4754 partitions to gs://gnomad-tmp-4day/rmc/rare_coding_exploded_context_in_hq_transcripts_regions.ht


In [28]:
context_ht = add_obs_annotation(context_ht)

INFO (constraint_utils 194): Adding observed annotation...


In [29]:
# Annotate O/E with all MC table – already takes care of transcript/RMC region concatenating
def get_oe_annotation_updated(ht, freeze: int):
    # Read in table with RMC OE where available, otherwise transcript OE
    # for each region/transcript
    rmc_ht = hl.read_table(
        "gs://regional_missense_constraint/temp/all_rmc_intervals.ht"
    )
    # Explode regions/transcripts to loci to join
    rmc_exploded = explode_intervals_to_loci(
        rmc_ht, interval_field="interval", keep_intervals=False
    ).key_by("locus", "transcript")
    # Join to annotate O/E
    ht = ht.annotate(oe=rmc_exploded[ht.locus, ht.transcript].section_oe)
    return ht

In [30]:
context_ht = get_oe_annotation_updated(context_ht, 2)

In [31]:
context_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'most_severe_consequence': str 
    'amino_acids': str 
    'codons': str 
    'lof': str 
    'lof_flags': str 
    'exomes_AN_percent': int32 
    'ref': str 
    'alt': str 
    'observed': int32 
    'oe': float64 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [32]:
from gnomad_constraint.resources.resource_utils import get_per_variant_expected_dataset

In [33]:
exp_ht = get_per_variant_expected_dataset(
    directory_post_fix="coverage_corrected", path_post_fix="coverage_corrected"
).ht()

In [34]:
exp_ht.show()

+---------------+------------+---------+-----+-----+-------------+------------+
| locus         | alleles    | context | ref | alt | was_flipped | transition |
+---------------+------------+---------+-----+-----+-------------+------------+
| locus<GRCh38> | array<str> | str     | str | str |        bool |       bool |
+---------------+------------+---------+-----+-----+-------------+------------+
| chr1:14414    | ["G","A"]  | "ACT"   | "C" | "T" |        True |       True |
| chr1:14414    | ["G","A"]  | "ACT"   | "C" | "T" |        True |       True |
| chr1:14414    | ["G","A"]  | "ACT"   | "C" | "T" |        True |       True |
| chr1:14414    | ["G","A"]  | "ACT"   | "C" | "T" |        True |       True |
| chr1:14414    | ["G","A"]  | "ACT"   | "C" | "T" |        True |       True |
| chr1:14414    | ["G","A"]  | "ACT"   | "C" | "T" |        True |       True |
| chr1:14414    | ["G","A"]  | "ACT"   | "C" | "T" |        True |       True |
| chr1:14414    | ["G","C"]  | "ACT"   | "C" | "G" |        True |      False |
| chr1:14414    | ["G","C"]  | "ACT"   | "C" | "G" |        True |      False |
| chr1:14414    | ["G","C"]  | "ACT"   | "C" | "G" |        True |      False |
+---------------+------------+---------+-----+-----+-------------+------------+

+-------+----------------------+---------------------+-------------------+
|   cpg | mutation_type        | mutation_type_model | methylation_level |
+-------+----------------------+---------------------+-------------------+
|  bool | str                  | str                 |             int32 |
+-------+----------------------+---------------------+-------------------+
| False | "non-CpG transition" | "non-CpG"           |                 0 |
| False | "non-CpG transition" | "non-CpG"           |                 0 |
| False | "non-CpG transition" | "non-CpG"           |                 0 |
| False | "non-CpG transition" | "non-CpG"           |                 0 |
| False | "non-CpG transition" | "non-CpG"           |                 0 |
| False | "non-CpG transition" | "non-CpG"           |                 0 |
| False | "non-CpG transition" | "non-CpG"           |                 0 |
| False | "transversion"       | "non-CpG"           |                 0 |
| False | "transversion"       | "non-CpG"           |                 0 |
| False | "transversion"       | "non-CpG"           |                 0 |
+-------+----------------------+---------------------+-------------------+

+-----------+----------------------+-------------------------------+
|      gerp | coverage.exomes.mean | coverage.exomes.median_approx |
+-----------+----------------------+-------------------------------+
|   float64 |              float64 |                         int32 |
+-----------+----------------------+-------------------------------+
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
| -3.08e-01 |             6.11e+00 |                             0 |
+-----------+----------------------+-------------------------------+

+-----------------------+--------------------------------+-----------+
| coverage.genomes.mean | coverage.genomes.median_approx | AN.exomes |
+-----------------------+--------------------------------+-----------+
|               float64 |                          int32 |     int64 |
+-----------------------+--------------------------------+-----------+
|              2.14e+01 |                             20 |

In [35]:
exp_ht.describe()

----------------------------------------
Global fields:
    'calculate_mu_globals': struct {
        freq_meta: array<dict<str, str>>, 
        ac_cutoff: int32, 
        min_cov: int32, 
        max_cov: int32, 
        gerp_lower_cutoff: float64, 
        gerp_upper_cutoff: float64, 
        genetic_ancestry_groups: array<str>, 
        downsampling_level: int32, 
        downsampling_idx: int32, 
        most_severe_consequence: array<str>
    } 
    'build_models_globals': struct {
        synonymous_transcript_filter_field: str, 
        low_cov_cutoff: int32, 
        high_cov_cutoff: int32, 
        upper_cov_cutoff: int32, 
        skip_coverage_model: bool
    } 
    'apply_models_globals': struct {
        low_cov_cutoff: int32, 
        high_cov_cutoff: int32, 
        skip_coverage_model: bool, 
        plateau_models: dict<struct {
            cpg: bool, 
            genomic_region: str
        }, array<array<float64>>>, 
        coverage_model: array<float64>, 
        lo

In [36]:
# :param variant_idx: Index of observed and expected arrays to use. Default is 0 (corresponds to counts calculated on gnomAD-wide / "global" frequencies).
variant_idx = 0

In [37]:
context_ht = context_ht.annotate(expected=exp_ht[context_ht.key].expected_variants[variant_idx])

In [38]:
context_ht = context_ht.key_by("locus", "alleles", "transcript")
context_ht = context_ht.select(
    "ref",
    "alt",
    "observed",
    "expected",
    "codons",
    "amino_acids",
    "oe",
)

In [39]:
def amino_acids_oe_path(
    is_train: bool = False,
    freeze: int = CURRENT_FREEZE,
) -> str:
    """
    Return path to Table containing all possible amino acid substitutions and their missense OE ratio.

    Table is input to missense badness calculations.

    :param int freeze: RMC data freeze number. Default is CURRENT_FREEZE.
    :return: Path to Table.
    """
    return f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{freeze}/{'train/' if is_train else ''}amino_acid_oe.ht"

In [40]:
amino_acids_oe_path()

'gs://regional_missense_constraint/MPC/4.1/2/amino_acid_oe.ht'

In [41]:
print("Writing out HT for all pass transcripts...")
context_ht = context_ht.checkpoint(
    amino_acids_oe_path(is_train=False),
    _read_if_exists=False,
    overwrite=True,
)

Writing out HT for all pass transcripts...


2026-01-20 03:32:37.044 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 03:33:46.584 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 03:41:57.535 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 03:50:56.037 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 03:56:56.620 Hail: INFO: wrote table with 86414950 rows in 4754 partitions to gs://regional_missense_constraint/MPC/4.1/2/amino_acid_oe.ht


In [55]:
print("Writing out HT for training transcripts only...")
train_transcripts = hl.experimental.read_expression(
    train_val_test_transcripts_path(fold=fold)
)
train_context_ht = context_ht.filter(train_transcripts.contains(context_ht.transcript))
train_context_ht.write(amino_acids_oe_path(is_train=True), overwrite=True)

Writing out HT for training transcripts only...


2025-12-08 04:17:54.044 Hail: INFO: wrote table with 69244666 rows in 4754 partitions to gs://regional_missense_constraint/MPC/4.1/2/train/amino_acid_oe.ht


# Calculate missense badness

In [42]:
def misbad_path(
    is_train: bool = False,
    freeze: int = CURRENT_FREEZE,
) -> str:
    """
    Table containing all possible amino acid substitutions and their missense badness scores.
    
    :param int freeze: RMC data freeze number. Default is CURRENT_FREEZE.
    :return: Path to Table.
    """
    return f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{freeze}/{'train/' if is_train else ''}missense_badness.ht"

In [43]:
import logging
logging.basicConfig(
    format="%(asctime)s (%(name)s %(lineno)s): %(message)s",
    datefmt="%m/%d/%Y %I:%M:%S %p",
)
logger = logging.getLogger("missense_badness")
logger.setLevel(logging.INFO)

In [44]:
def calculate_misbad_updated(
    use_exac_oe_cutoffs: bool,
    overwrite_temp: bool,
    oe_threshold: float = 0.6,
    is_train: bool = False,
    freeze: int = CURRENT_FREEZE,
) -> None:
    """
    Calculate missense badness scores using Table with all amino acid substitutions and their missense observed/expected (OE) ratio.

    If `use_exac_oe_cutoffs` is set, will remove all rows with 0.6 < OE <= 0.8.

    .. note::
        Assumes table(s) containing all possible amino acid substitutions and their missense OE ratio exists.

    :param bool use_exac_oe_cutoffs: Whether to use the same missense OE cutoffs as in ExAC missense badness calculation.
    :param bool overwrite_temp: Whether to overwrite intermediate temporary data if it already exists.
        If False, will read existing intermediate temporary data rather than overwriting.
    :param float oe_threshold: OE Threshold used to split Table.
        Rows with OE less or equal to this threshold will be considered "low" OE, and
        rows with OE greater than this threshold will considered "high" OE.
        Default is 0.6.
    :param int freeze: RMC freeze number. Default is CURRENT_FREEZE.
    :return: None; writes Table(s) with missense badness scores to resource path(s).
    """
    aa_oe_ht_path = amino_acids_oe_path(is_train=is_train, freeze=freeze)
    if not file_exists(aa_oe_ht_path):
        raise DataException(
            "Table with all amino acid substitutions and missense OE doesn't exist!"
        )
    
    ht = hl.read_table(aa_oe_ht_path)

    # Filter with ExAC O/E cutoffs if specified
    if use_exac_oe_cutoffs:
        logger.info("Removing rows with OE greater than 0.6 and less than 0.8...")
        ht = ht.filter((ht.oe <= 0.6) | (ht.oe > 0.8))

    logger.info(
        "Splitting input Table by OE to get synonymous and nonsense rates for high"
        " and low OE groups..."
    )
    oe_hts = {}
    for keep_high_oe in [True, False]:
        oe_direction = "high" if keep_high_oe else "low"
        logger.info(
            "Creating %s missense OE (OE > %s) HT...", oe_direction, oe_threshold
        )
        oe_hts[oe_direction] = aggregate_aa_and_filter_oe(
            ht, keep_high_oe=keep_high_oe
        )
        oe_hts[oe_direction] = oe_hts[oe_direction].checkpoint(
            f"{TEMP_PATH_WITH_FAST_DEL}/amino_acids_{oe_direction}_oe{'_train' if is_train else ''}.ht",
            _read_if_exists=not overwrite_temp,
            overwrite=overwrite_temp,
        )

    logger.info("Re-joining split HTs to calculate missense badness...")
    mb_ht = oe_hts["high"].join(oe_hts["low"], how="outer")
    mb_ht = mb_ht.transmute(mut_type=hl.coalesce(mb_ht.mut_type, mb_ht.mut_type_1))
    mb_ht = mb_ht.annotate(
        high_low=(
            (mb_ht.high_obs / mb_ht.high_pos) / (mb_ht.low_obs / mb_ht.low_pos)
        )
    )

    total_rates = {}
    for mut_type in ["syn", "non"]:
        logger.info("Calculating %s rates...", mut_type)
        total_rates[mut_type] = (
            get_total_csq_count(
                oe_hts["high"], csq=mut_type, count_field="high_obs"
            )
            / get_total_csq_count(
                oe_hts["high"], csq=mut_type, count_field="high_pos"
            )
        ) / (
            get_total_csq_count(oe_hts["low"], csq=mut_type, count_field="low_obs")
            / get_total_csq_count(
                oe_hts["low"], csq=mut_type, count_field="low_pos"
            )
        )
        logger.info("%s rate: %f", mut_type, total_rates[mut_type])

    logger.info("Calculating missense badness...")
    mb_ht = mb_ht.annotate(
        misbad=hl.or_missing(
            mb_ht.mut_type == "mis",
            # Cap missense badness at 1
            hl.min(
                (mb_ht.high_low - total_rates["syn"])
                / (total_rates["non"] - total_rates["syn"]),
                1,
            ),
        ),
    )
    mb_ht = mb_ht.naive_coalesce(1)

    mb_ht.write(misbad_path(is_train=is_train, freeze=freeze), overwrite=True)

In [45]:
# Use bottom 10% vs. top 90% of O/E
pass_coding_locus_oe_pctiles = hl.experimental.read_expression(
    "gs://regional_missense_constraint/temp/v4_freeze2_high_cov_pass_coding_locus_oe_pctiles.he"
)

In [46]:
hl.eval(pass_coding_locus_oe_pctiles)

[0.2395464826087203,
 0.3432136282005,
 0.4073293863937911,
 0.45457342510264087,
 0.4990339825608904,
 0.532280388194433,
 0.5643044074206567,
 0.5940522201666877,
 0.6167181223782818,
 0.638936575540817,
 0.6586161623491652,
 0.6773746344927583,
 0.6924764671960655,
 0.7070256780184616,
 0.7209926302935349,
 0.73317422782763,
 0.7450898634525336,
 0.7559315536778858,
 0.7669027078657205,
 0.7766240131821815,
 0.7870337473091297,
 0.7960420058701309,
 0.8043447936631972,
 0.8105400332504247,
 0.8173873166831267,
 0.8247911036367749,
 0.8307962152386414,
 0.8368202952107673,
 0.8430540532420333,
 0.848668475944845,
 0.8533736020727949,
 0.858798486832439,
 0.8635993737284111,
 0.8684303569498532,
 0.8725088137726166,
 0.8771381275205778,
 0.8808419948483429,
 0.8844130384390697,
 0.8876102079202929,
 0.8913135302311965,
 0.8942986729396945,
 0.8976546212682023,
 0.9010592079990677,
 0.9044193053902104,
 0.9082054569375245,
 0.911324785530462,
 0.914491703461472,
 0.9175379569905363,
 0

In [47]:
len(hl.eval(pass_coding_locus_oe_pctiles))

100

In [48]:
hl.eval(pass_coding_locus_oe_pctiles)[9]

0.638936575540817

In [49]:
from rmc.utils.missense_badness import (
    aggregate_aa_and_filter_oe,
    get_total_csq_count,
)

In [50]:
calculate_misbad_updated(
    use_exac_oe_cutoffs=False,
    overwrite_temp=True,
    oe_threshold=hl.eval(pass_coding_locus_oe_pctiles)[9],
    is_train=False,
    freeze=CURRENT_FREEZE,
)

INFO (missense_badness 39): Splitting input Table by OE to get synonymous and nonsense rates for high and low OE groups...
INFO (missense_badness 46): Creating high missense OE (OE > 0.638936575540817) HT...
INFO (calculate_missense_badness 283): Filtering HT on missense OE values...
INFO (calculate_missense_badness 287): Grouping HT and aggregating observed and possible variant counts...
INFO (calculate_missense_badness 294): Adding variant consequence (mut_type) annotation and returning...
2026-01-20 03:57:09.329 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 03:57:43.076 Hail: INFO: wrote table with 185 rows in 178 partitions to gs://gnomad-tmp-4day/rmc/amino_acids_high_oe.ht
INFO (missense_badness 46): Creating low missense OE (OE > 0.638936575540817) HT...
INFO (calculate_missense_badness 283): Filtering HT on missense OE values...
INFO (calculate_missense_badness 287): Grouping HT and aggregating observed and possible variant counts...
INFO (calculate_misse

In [51]:
hl.read_table("gs://regional_missense_constraint/MPC/4.1/2/missense_badness.ht").show()

,,,,,,,,
ref,alt,high_obs,high_pos,mut_type,low_obs,low_pos,high_low,misbad
str,str,int64,int64,str,int64,int64,float64,float64
"""Ala""","""Ala""",426108,1793232,"""syn""",30210,148394,1.17e+00,NA
"""Ala""","""Asp""",56657,368463,"""mis""",1354,31955,3.63e+00,1.00e+00
"""Ala""","""Glu""",39347,182465,"""mis""",989,14995,3.27e+00,1.00e+00
"""Ala""","""Gly""",84061,640486,"""mis""",3123,52728,2.22e+00,6.93e-01
"""Ala""","""Pro""",68704,640940,"""mis""",1704,52943,3.33e+00,1.00e+00
"""Ala""","""Ser""",126578,568964,"""mis""",5312,49280,2.06e+00,5.97e-01
"""Ala""","""Thr""",267551,587392,"""mis""",11397,48310,1.93e+00,5.14e-01
"""Ala""","""Val""",262129,588250,"""mis""",10364,48384,2.08e+00,6.08e-01


In [66]:
calculate_misbad_updated(
    use_exac_oe_cutoffs=False,
    overwrite_temp=True,
    do_k_fold_training=True,
    oe_threshold=hl.eval(pass_coding_locus_oe_pctiles)[9],
    freeze=CURRENT_FREEZE,
)

INFO (missense_badness 67): Splitting input Table by OE to get synonymous and nonsense rates for high and low OE groups...
INFO (missense_badness 74): Creating high missense OE (OE > 0.638936575540817) HT...
INFO (calculate_missense_badness 283): Filtering HT on missense OE values...
INFO (calculate_missense_badness 287): Grouping HT and aggregating observed and possible variant counts...
INFO (calculate_missense_badness 294): Adding variant consequence (mut_type) annotation and returning...
2025-12-08 04:23:06.225 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-12-08 04:23:30.736 Hail: INFO: wrote table with 185 rows in 178 partitions to gs://gnomad-tmp-4day/rmc/amino_acids_high_oe_train_fold1.ht
INFO (missense_badness 74): Creating low missense OE (OE > 0.638936575540817) HT...
INFO (calculate_missense_badness 283): Filtering HT on missense OE values...
INFO (calculate_missense_badness 287): Grouping HT and aggregating observed and possible variant counts...
INFO (cal

INFO (calculate_missense_badness 294): Adding variant consequence (mut_type) annotation and returning...
2025-12-08 04:28:01.130 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-12-08 04:28:25.214 Hail: INFO: wrote table with 185 rows in 178 partitions to gs://gnomad-tmp-4day/rmc/amino_acids_high_oe_train_fold5.ht
INFO (missense_badness 74): Creating low missense OE (OE > 0.638936575540817) HT...
INFO (calculate_missense_badness 283): Filtering HT on missense OE values...
INFO (calculate_missense_badness 287): Grouping HT and aggregating observed and possible variant counts...
INFO (calculate_missense_badness 294): Adding variant consequence (mut_type) annotation and returning...
2025-12-08 04:28:31.754 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-12-08 04:28:47.485 Hail: INFO: wrote table with 185 rows in 178 partitions to gs://gnomad-tmp-4day/rmc/amino_acids_low_oe_train_fold5.ht
INFO (missense_badness 86): Re-joining split HTs to calculate missense 

In [67]:
hl.read_table("gs://regional_missense_constraint/MPC/4.1/2/train/missense_badness.ht").show()

,,,,,,,,
ref,alt,high_obs,high_pos,mut_type,low_obs,low_pos,high_low,misbad
str,str,int64,int64,str,int64,int64,float64,float64
"""Ala""","""Ala""",341911,1438099,"""syn""",24001,117605,1.16e+00,NA
"""Ala""","""Asp""",45501,295713,"""mis""",1093,25318,3.56e+00,1.00e+00
"""Ala""","""Glu""",31580,146355,"""mis""",810,11885,3.17e+00,1.00e+00
"""Ala""","""Gly""",67418,513534,"""mis""",2492,41825,2.20e+00,7.03e-01
"""Ala""","""Pro""",55072,513993,"""mis""",1377,41976,3.27e+00,1.00e+00
"""Ala""","""Ser""",101716,456276,"""mis""",4266,38996,2.04e+00,5.96e-01
"""Ala""","""Thr""",214692,471115,"""mis""",9076,38258,1.92e+00,5.21e-01
"""Ala""","""Val""",209911,471601,"""mis""",8207,38286,2.08e+00,6.21e-01
